In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from utils.modelling import modeling_prep

### 1. Build model input

Satu panggilan — urutan langkahnya didefinisikan di `utils/modeling_prep.py`,
bukan di notebook ini.

In [2]:
model_input = modeling_prep.build_model_input()
print(f"{len(model_input):,} baris × {len(model_input.columns)} kolom")
model_input[["is_event_driven", "demand_segment", "fold_id"]].head()

1,522,868 baris × 75 kolom


,is_event_driven,demand_segment,fold_id
0,False,smooth,NaN
1,False,smooth,NaN
2,False,smooth,NaN
3,False,smooth,NaN
4,False,smooth,NaN


### 2. Segment & fold QA

In [3]:
pair_segment = model_input.groupby(modeling_prep.PAIR_COLS, observed=True)["demand_segment"].first()
print(pair_segment.value_counts(), "\n")
print(model_input["fold_id"].value_counts(dropna=False).sort_index())

assert model_input.loc[model_input["Tanggal"] >= "2025-12-01", "fold_id"].isna().all(), \
    "Baris Desember tidak boleh punya fold — itu test set terkunci"
print("\n✓ Desember bebas fold")

demand_segment
intermittent    1309
lumpy            945
erratic          405
smooth           320
Name: count, dtype: int64 

fold_id
1.0      77318
2.0      76266
3.0      71066
4.0      69510
5.0      61407
NaN    1167301
Name: count, dtype: int64

✓ Desember bebas fold


### 3. Imputation QA

Cek yang paling penting: kolom kedekatan event tidak boleh terisi `0`
(itu berarti "hari ini Idul Fitri"), dan sentinel harus di atas 70
(`days_until_ramadan` mencapai 70 di data asli).

In [4]:
for col in modeling_prep.EVENT_PROXIMITY_COLS:
    assert model_input[col].notna().all(), f"{col} masih punya null"

filled = (model_input["days_until_ramadan"] == modeling_prep.EVENT_PROXIMITY_SENTINEL).mean()
print(f"days_until_ramadan sentinel: {filled:.1%} baris")
assert modeling_prep.EVENT_PROXIMITY_SENTINEL > 70, "Sentinel bertabrakan dengan nilai asli"

print(model_input[["was_relocated", "has_baseline"]].mean().round(3))
print("\n✓ imputasi aman")

days_until_ramadan sentinel: 84.6% baris
was_relocated    0.156
has_baseline     0.856
dtype: float64

✓ imputasi aman


### 4. Adapter contract

Ini yang membuat perbandingan XGBoost / Random Forest / LSTM bisa
dipertanggungjawabkan: kedua adapter wajib melihat baris yang sama persis.

In [5]:
feature_cols = [
    "lag_1", "lag_7", "lag_28", "roll_mean_7", "roll_mean_28", "roll_std_7",
    "day_of_week", "is_weekend", "is_national_holiday", "lead_time_days",
    "days_until_ramadan", "days_since_relocation", "was_relocated",
    "baseline_ratio", "has_baseline", "is_event_driven",
    "Kode Barang_idx", "Nama Cabang_idx", "kota_idx", "demand_segment_idx",
]

sample_branches = model_input["Nama Cabang"].unique()[:3]
sample = model_input[model_input["Nama Cabang"].isin(sample_branches)]

tabular = modeling_prep.to_tabular(sample, feature_cols=feature_cols)
sequences = modeling_prep.to_sequences(sample, feature_cols=feature_cols)
modeling_prep.validate_contract(tabular, sequences)

print(f"tabular X : {tabular['X'].shape}")
print(f"sequence X: {sequences['X'].shape}")
print("✓ kontrak lolos")

tabular X : (97824, 20)
sequence X: (97824, 28, 20)
✓ kontrak lolos


### 5. Export

In [6]:
modeling_prep.export_model_input(model_input)
print(f"✓ ditulis ke {modeling_prep.MODEL_INPUT_FILE}")

✓ ditulis ke /Users/ramapdp/Project/Personal/forecast-scm/dataset/model_ready/model_input.parquet
